Preprocessing 

In [7]:
import pandas as pd
import numpy as np



train= pd.read_csv('claims_train.csv')
test = pd.read_csv('claims_test.csv')

#Cleaning weird values that found during cleaning
train = train[train['Exposure'] <= 1].copy()
test =test[test['Exposure']<=1].copy()

# Adding Risk column
train['Risk'] = train['ClaimNb'] / train['Exposure'] 
test['Risk'] =  test['ClaimNb'] / test['Exposure']

#Encoding
train_encoded=pd.get_dummies(train, columns=['VehBrand', 'VehGas', 'Region'], drop_first=True)#Encoding categorical values
area_map={'A':1,'B':2,'C':3,'D':4,'E':5,'F':6}
train_encoded['Area']=train_encoded['Area'].map(area_map)

test_encoded=pd.get_dummies(test, columns=['VehBrand', 'VehGas', 'Region'], drop_first=True)#Encoding categorical values
area_map={'A':1,'B':2,'C':3,'D':4,'E':5,'F':6}
test_encoded['Area']=test_encoded['Area'].map(area_map)

w=train['Exposure']
X_train=train_encoded.drop(columns=['ClaimNb','Exposure', 'IDpol','Risk'])
y_train=train_encoded['ClaimNb']

X_test=test_encoded.drop(columns=['ClaimNb','Exposure', 'IDpol','Risk'])
y_test=test_encoded['ClaimNb']


Training the model 

In [16]:
import xgboost as xgb
from sklearn.metrics import  recall_score, precision_score

xgb = xgb.XGBRegressor(
    objective="count:poisson",
    n_estimators=400,
    learning_rate=0.05,
    max_depth=6,
    subsample=0.8,
    colsample_bytree=0.8
)

xgb.fit(X_train, y_train, sample_weight=w)
y_pred=xgb.predict(X_test)

from sklearn.metrics import mean_poisson_deviance
eps = 1e-9
y_pred_safe = np.clip(y_pred, eps, None)
mpd_model = mean_poisson_deviance(y_test, y_pred_safe)

print("Model Poisson Deviance:", mpd_model)

df=pd.DataFrame({"y_test":y_test, "y_pred":y_pred})
df["bucket"] = pd.qcut(df.y_pred, q=10, labels=False)

df.groupby("bucket")["y_test"].mean()


Model Poisson Deviance: 0.3073713481426239


bucket
0    0.023563
1    0.028662
2    0.030213
3    0.039817
4    0.039518
5    0.045874
6    0.052818
7    0.057250
8    0.073275
9    0.146561
Name: y_test, dtype: float64